# Tarea A --- Preprocesamiento Tabular

**Proyecto:** Itaca SmartDiag --- Capstone Samsung Innovation Campus
**Responsable:** Jorge Rodriguez
**Semana:** 1 | **Stage:** 2

Convierte las variables estructuradas de `data/diagnosticos_itaca_clean.csv`
en arreglos numericos para la rama tabular de la DNN, genera los splits
train/val/test compartidos por todo el equipo y serializa los artefactos
de preprocesamiento (scaler, encoder, mapeo de clases) que usara el backend.

Ver especificacion completa en `TASK_A_TABULAR.md`.

## 0. Configuracion y constantes

In [1]:
# Librerias permitidas por el spec: pandas, numpy, scikit-learn, joblib
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Rutas como constantes al inicio del script (regla del proyecto: sin argparse).
# El notebook se ejecuta con working directory = preprocessing/.
CLEAN_DATA_PATH = "../data/diagnosticos_itaca_clean.csv"

SPLITS_DIR = "splits"
ARTIFACTS_DIR = "artifacts"
OUTPUT_DIR = "output"

TRAIN_PATH = f"{SPLITS_DIR}/train.csv"
VAL_PATH = f"{SPLITS_DIR}/val.csv"
TEST_PATH = f"{SPLITS_DIR}/test.csv"

SCALER_PATH = f"{ARTIFACTS_DIR}/scaler.joblib"
ENCODER_PATH = f"{ARTIFACTS_DIR}/onehot_encoder.joblib"
CLASS_MAP_PATH = f"{ARTIFACTS_DIR}/class_map.json"
CLASS_WEIGHTS_PATH = f"{ARTIFACTS_DIR}/class_weights.json"

# Semilla fija exigida por el spec en TODA operacion aleatoria (seccion 7).
RANDOM_STATE = 42

# Proporciones de particion definidas en la seccion 4.1 del spec:
# train 75%, val 15%, test 10%. temp = 25% del total; dentro de temp,
# val es 15/25 = 60% de temp y test es 10/25 = 40% de temp.
TRAIN_SIZE = 0.75
VAL_SIZE_OF_TEMP = 0.60

TARGET_COL = "nivel_madurez"

# Variables numericas de la rama tabular (seccion 4.2 del spec).
NUMERIC_COLS = ["porcentaje_procesos_documentados", "presupuesto_anual_tecnologia"]

# Variables categoricas de la rama tabular (seccion 4.3 del spec).
CATEGORICAL_COLS = ["sector", "tamano_empresa"]

# Mapeo EXACTO del target exigido por la seccion 4.4 del spec: no se
# cambia el orden, todo el equipo (modelo y backend) depende de el.
CLASS_MAP = {"Inicial": 0, "En Desarrollo": 1, "Definido": 2, "Optimizado": 3}

## 4.1 Particiones estratificadas (train / val / test)

Bloquea todo lo demas: el resto de las tareas (escalado, one-hot, target,
matrices) parte de estos tres conjuntos. Se usa `train_test_split` en dos
pasos, siempre estratificando por `nivel_madurez` para preservar la
proporcion de clases (la clase `Optimizado` es minoritaria, ~13%).

In [2]:
# Carga del dataset limpio (solo lectura, no se modifica en esta tarea).
df = pd.read_csv(CLEAN_DATA_PATH, encoding="utf-8")

print(f"Filas cargadas: {len(df)}")
print(f"Columnas ({len(df.columns)}): {list(df.columns)}")

df.head()

Filas cargadas: 3000
Columnas (8): ['id_diagnostico', 'sector', 'tamano_empresa', 'porcentaje_procesos_documentados', 'presupuesto_anual_tecnologia', 'respuesta_texto', 'nivel_madurez', 'recomendacion_principal']


,id_diagnostico,sector,tamano_empresa,porcentaje_procesos_documentados,presupuesto_anual_tecnologia,respuesta_texto,nivel_madurez,recomendacion_principal
0,DIAG_0000,Tecnologia,Micro,0.05,3000000,"El trabajo es muy empírico, no hay documentaci...",Inicial,Mapear flujos de trabajo de desarrollo y usar ...
1,DIAG_0001,Tecnologia,Grande,0.81,261000000,"Todo está automatizado, usamos datos para mejo...",Optimizado,Aplicar MLOps y analítica predictiva para opti...
2,DIAG_0002,Tecnologia,Pequena,0.27,47000000,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
3,DIAG_0003,Tecnologia,Pequena,0.30,19000000,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
4,DIAG_0004,Manufactura,Micro,0.08,6000000,"Todo es manual, dependemos de una sola persona...",Inicial,Estandarizar el registro diario de mermas e in...


In [3]:
# Paso 1: separar train (75%) de un conjunto temporal (25%) que luego se
# divide en val y test. Estratificado por el target para preservar la
# proporcion de clases en cada conjunto.
df_train, df_temp = train_test_split(
    df,
    train_size=TRAIN_SIZE,
    stratify=df[TARGET_COL],
    random_state=RANDOM_STATE,
)

print(f"train: {len(df_train)} filas ({len(df_train) / len(df):.1%})")
print(f"temp:  {len(df_temp)} filas ({len(df_temp) / len(df):.1%})")

train: 2250 filas (75.0%)
temp:  750 filas (25.0%)


In [4]:
# Paso 2: dividir temp (25% del total) en val (60% de temp -> 15% del
# total) y test (40% de temp -> 10% del total). Se estratifica sobre el
# nivel de madurez DENTRO de temp, no sobre el dataset completo.
df_val, df_test = train_test_split(
    df_temp,
    train_size=VAL_SIZE_OF_TEMP,
    stratify=df_temp[TARGET_COL],
    random_state=RANDOM_STATE,
)

print(f"val:  {len(df_val)} filas ({len(df_val) / len(df):.1%})")
print(f"test: {len(df_test)} filas ({len(df_test) / len(df):.1%})")

val:  450 filas (15.0%)
test: 300 filas (10.0%)


### Validaciones (criterios de aceptacion 1 y 2 del spec)

In [5]:
# Criterio de aceptacion 1: los tres splits suman 3000 filas y no
# comparten ningun id_diagnostico entre si.
total_rows = len(df_train) + len(df_val) + len(df_test)
assert total_rows == len(df), f"Los splits no suman el total: {total_rows} != {len(df)}"

ids_train = set(df_train["id_diagnostico"])
ids_val = set(df_val["id_diagnostico"])
ids_test = set(df_test["id_diagnostico"])

assert ids_train.isdisjoint(ids_val), "train y val comparten id_diagnostico"
assert ids_train.isdisjoint(ids_test), "train y test comparten id_diagnostico"
assert ids_val.isdisjoint(ids_test), "val y test comparten id_diagnostico"

print(f"Total filas en los tres splits: {total_rows} (esperado: {len(df)})")
print("Ids sin solapamiento entre splits: OK")

Total filas en los tres splits: 3000 (esperado: 3000)
Ids sin solapamiento entre splits: OK


In [6]:
# Criterio de aceptacion 2: la distribucion porcentual de nivel_madurez
# debe diferir en menos de 1 punto porcentual entre train, val y test
# (evidencia de que la estratificacion funciono).
def class_pct(frame: pd.DataFrame) -> pd.Series:
    return (100 * frame[TARGET_COL].value_counts(normalize=True)).round(2)

pct_table = pd.DataFrame({
    "train": class_pct(df_train),
    "val": class_pct(df_val),
    "test": class_pct(df_test),
}).sort_index()

print("Distribucion porcentual de nivel_madurez por split:")
print(pct_table)

max_diff = (pct_table.max(axis=1) - pct_table.min(axis=1)).max()
print(f"\nMaxima diferencia entre splits: {max_diff:.2f} puntos porcentuales")

assert max_diff < 1.0, "La estratificacion no cumple el criterio de <1pp de diferencia"
print("Estratificacion validada: diferencia < 1 punto porcentual entre splits")

Distribucion porcentual de nivel_madurez por split:
               train    val   test
nivel_madurez                     
Definido       30.36  30.44  30.33
En Desarrollo  30.80  30.67  30.67
Inicial        26.09  26.00  26.33
Optimizado     12.76  12.89  12.67

Maxima diferencia entre splits: 0.33 puntos porcentuales
Estratificacion validada: diferencia < 1 punto porcentual entre splits


### Guardado de los splits

In [7]:
# Guardar los tres splits COMPLETOS (las 8 columnas originales, incluyendo
# id_diagnostico y respuesta_texto que la Tarea B necesita) en UTF-8.
df_train.to_csv(TRAIN_PATH, index=False, encoding="utf-8")
df_val.to_csv(VAL_PATH, index=False, encoding="utf-8")
df_test.to_csv(TEST_PATH, index=False, encoding="utf-8")

print(f"Guardado: {TRAIN_PATH} ({len(df_train)} filas)")
print(f"Guardado: {VAL_PATH} ({len(df_val)} filas)")
print(f"Guardado: {TEST_PATH} ({len(df_test)} filas)")

Guardado: splits/train.csv (2250 filas)
Guardado: splits/val.csv (450 filas)
Guardado: splits/test.csv (300 filas)


## 4.2 Escalado de variables numericas

Escala `porcentaje_procesos_documentados` y `presupuesto_anual_tecnologia`
con `StandardScaler`. **Regla anti data-leakage (seccion 7 del spec):** el
scaler se ajusta (`fit`) UNICAMENTE con los datos de train; a val y test
solo se les aplica `transform`, nunca `fit` ni `fit_transform`.

In [8]:
# Ajustar (fit) el StandardScaler UNICAMENTE con las columnas numericas
# de train. Val y test NUNCA participan del fit.
scaler = StandardScaler()
scaler.fit(df_train[NUMERIC_COLS])

print(f"Scaler ajustado sobre columnas: {NUMERIC_COLS}")
print(f"Media aprendida (train): {scaler.mean_}")
print(f"Desviacion estandar aprendida (train): {scaler.scale_}")

Scaler ajustado sobre columnas: ['porcentaje_procesos_documentados', 'presupuesto_anual_tecnologia']
Media aprendida (train): [4.45502222e-01 8.36982222e+07]
Desviacion estandar aprendida (train): [2.79534281e-01 1.10588434e+08]


In [9]:
# A val y test solo se les aplica transform, usando las estadisticas
# (media, desviacion) aprendidas exclusivamente de train.
X_num_train = scaler.transform(df_train[NUMERIC_COLS])
X_num_val = scaler.transform(df_val[NUMERIC_COLS])
X_num_test = scaler.transform(df_test[NUMERIC_COLS])

print(f"X_num_train shape: {X_num_train.shape}")
print(f"X_num_val shape:   {X_num_val.shape}")
print(f"X_num_test shape:  {X_num_test.shape}")

X_num_train shape: (2250, 2)
X_num_val shape:   (450, 2)
X_num_test shape:  (300, 2)


In [10]:
# Criterio de aceptacion 3 (parcial): en train, la media de cada columna
# escalada debe estar cerca de 0 y la desviacion estandar cerca de 1.
train_means = X_num_train.mean(axis=0)
train_stds = X_num_train.std(axis=0)

print("Media de columnas escaladas en train (esperado ~0):", np.round(train_means, 4))
print("Desviacion estandar en train (esperado ~1):", np.round(train_stds, 4))

assert np.allclose(train_means, 0, atol=1e-8), "Las medias de train no estan cerca de 0"
assert np.allclose(train_stds, 1, atol=1e-8), "Las desviaciones de train no estan cerca de 1"
print("Escalado validado: medias ~0 y desviaciones ~1 en train")

Media de columnas escaladas en train (esperado ~0): [-0. -0.]
Desviacion estandar en train (esperado ~1): [1. 1.]
Escalado validado: medias ~0 y desviaciones ~1 en train


In [11]:
# Guardar el scaler ajustado: el backend lo reutilizara en produccion
# para aplicar la MISMA transformacion que se uso en entrenamiento.
joblib.dump(scaler, SCALER_PATH)
print(f"Guardado: {SCALER_PATH}")

Guardado: artifacts/scaler.joblib


## 4.3 Codificacion one-hot de categoricas

Codifica `sector` (4 categorias) y `tamano_empresa` (4 categorias) con
`OneHotEncoder(sparse_output=False, handle_unknown="ignore")`, resultando
en 8 columnas binarias. Misma regla anti data-leakage que en 4.2: `fit`
solo en train, `transform` en val/test.

In [12]:
# Ajustar (fit) el OneHotEncoder UNICAMENTE con las columnas categoricas
# de train. Val y test NUNCA participan del fit. handle_unknown="ignore"
# evita errores si en produccion llega una categoria no vista en train.
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoder.fit(df_train[CATEGORICAL_COLS])

# Documentar el orden de las columnas de salida (exigido por la seccion 4.3).
feature_names = encoder.get_feature_names_out(CATEGORICAL_COLS)
print(f"Encoder ajustado sobre columnas: {CATEGORICAL_COLS}")
print(f"Columnas de salida ({len(feature_names)}), en orden:")
for i, name in enumerate(feature_names):
    print(f"  [{i}] {name}")

Encoder ajustado sobre columnas: ['sector', 'tamano_empresa']
Columnas de salida (8), en orden:
  [0] sector_Manufactura
  [1] sector_Retail
  [2] sector_Servicios
  [3] sector_Tecnologia
  [4] tamano_empresa_Grande
  [5] tamano_empresa_Mediana
  [6] tamano_empresa_Micro
  [7] tamano_empresa_Pequena


In [13]:
# A val y test solo se les aplica transform, con las categorias
# aprendidas exclusivamente de train.
X_cat_train = encoder.transform(df_train[CATEGORICAL_COLS])
X_cat_val = encoder.transform(df_val[CATEGORICAL_COLS])
X_cat_test = encoder.transform(df_test[CATEGORICAL_COLS])

print(f"X_cat_train shape: {X_cat_train.shape}")
print(f"X_cat_val shape:   {X_cat_val.shape}")
print(f"X_cat_test shape:  {X_cat_test.shape}")

assert X_cat_train.shape[1] == 8, "Se esperaban 8 columnas one-hot (4 sectores + 4 tamanos)"
print("Numero de columnas one-hot validado: 8")

X_cat_train shape: (2250, 8)
X_cat_val shape:   (450, 8)
X_cat_test shape:  (300, 8)
Numero de columnas one-hot validado: 8


In [14]:
# Guardar el encoder ajustado: el backend lo reutilizara en produccion
# para aplicar la MISMA codificacion que se uso en entrenamiento.
joblib.dump(encoder, ENCODER_PATH)
print(f"Guardado: {ENCODER_PATH}")

Guardado: artifacts/onehot_encoder.joblib


## 4.4 Codificacion del target y class weights

Mapea `nivel_madurez` a enteros con el `CLASS_MAP`. 
Calcula pesos de clase balanceados usando
unicamente las etiquetas de train, para compensar el desbalance de la
clase `Optimizado` (~13% del dataset) durante el entrenamiento.

In [15]:
# Mapear nivel_madurez a enteros en los tres splits.
y_train = df_train[TARGET_COL].map(CLASS_MAP).astype("int64").to_numpy()
y_val = df_val[TARGET_COL].map(CLASS_MAP).astype("int64").to_numpy()
y_test = df_test[TARGET_COL].map(CLASS_MAP).astype("int64").to_numpy()

print(f"CLASS_MAP: {CLASS_MAP}")
print(f"y_train shape: {y_train.shape}, clases presentes: {sorted(set(y_train))}")
print(f"y_val shape:   {y_val.shape}")
print(f"y_test shape:  {y_test.shape}")

CLASS_MAP: {'Inicial': 0, 'En Desarrollo': 1, 'Definido': 2, 'Optimizado': 3}
y_train shape: (2250,), clases presentes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
y_val shape:   (450,)
y_test shape:  (300,)


In [ ]:
# Guardar el mapeo de clases
with open(CLASS_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump(CLASS_MAP, f, ensure_ascii=False, indent=2)

with open(CLASS_MAP_PATH, "r", encoding="utf-8") as f:
    loaded_map = json.load(f)
assert loaded_map == CLASS_MAP, "El class_map.json guardado no coincide con CLASS_MAP"

print(f"Guardado y verificado: {CLASS_MAP_PATH}")

Guardado y verificado: artifacts/class_map.json


In [ ]:
# Pesos de clase "balanced": se calculan unicamente con las etiquetas de
# train (val y test no deben influir en absoluto en los pesos).
class_labels = np.array(sorted(CLASS_MAP.values()))
weights = compute_class_weight(class_weight="balanced", classes=class_labels, y=y_train)

class_weights = {str(int(c)): round(float(w), 4) for c, w in zip(class_labels, weights)}
print("Pesos de clase (balanced, calculados solo con train):")
for k, v in class_weights.items():
    print(f"  {k}: {v}")

Pesos de clase (balanced, calculados solo con train):
  0: 0.9583
  1: 0.8117
  2: 0.8236
  3: 1.9599


In [ ]:
# Guardar los pesos de clase con claves enteras en string
with open(CLASS_WEIGHTS_PATH, "w", encoding="utf-8") as f:
    json.dump(class_weights, f, ensure_ascii=False, indent=2)

print(f"Guardado: {CLASS_WEIGHTS_PATH}")

Guardado: artifacts/class_weights.json
